<a href="https://colab.research.google.com/github/abeeraz379/Clickstream-prediction-for-Online-Shopping-Project/blob/main/Clickstream-DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clickstream Data for Online Shopping Project

- Data Link : https://archive.ics.uci.edu/dataset/553/clickstream+data+for+online+shopping




## Data Dictionary
Data description “e-shop clothing 2008”

Variables:

1. YEAR (2008)
2. MONTH -> from April (4) to August (8)
3. DAY -> day number of the month
4. ORDER -> sequence of clicks during one session
5. COUNTRY -> variable indicating the country of origin of the IP address
6. SESSION ID -> variable indicating session id (short record)
7. PAGE 1 (MAIN CATEGORY) -> concerns the main product category
8. PAGE 2 (CLOTHING MODEL) -> contains information about the code for each product
(217 products)
9. COLOUR -> colour of product
10. LOCATION -> photo location on the page, the screen has been divided into six parts
11. MODEL PHOTOGRAPHY -> variable with two categories
12. PRICE -> price in US dollars
13. PRICE 2 -> variable informing whether the price of a particular product is higher than
the average price for the entire product category
14. PAGE -> page number within the e-store website (from 1 to 5)


## The Target is **order**
- Regression task
- The goal is to increase no. of clicks during one session


# Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import seaborn as sns

from sklearn.preprocessing import StandardScaler,OrdinalEncoder,LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn import set_config
set_config(transform_output='pandas')

import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.activations import relu

import warnings
warnings.filterwarnings('ignore')

# Load and inspect data

In [ ]:
df =pd.read_csv('/content/drive/MyDrive/NLP/e_shop_clothing_2008.csv',sep=";")
df.head()

In [ ]:
df.info()

## unique values for each feature

In [ ]:
# show each feature with his values
for col in df.columns:
  print(f"{col} : {df[col].unique()}")

## check Missing values

In [ ]:
df.isna().sum()

## check duplicated

In [ ]:
df.duplicated().sum()

## Label Encoder

In [ ]:
le = LabelEncoder()
df['page 2 (clothing model)'] = le.fit_transform(df['page 2 (clothing model)'])



# Data visualization

In [ ]:
# Creatre heatmap
plt.figure(figsize=(10,10))

num_cols = df.select_dtypes(include=['number']).columns
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.show()

## Distribution

In [ ]:
# create a distribution plot for each feature in df
for col in df.columns:
    sns.displot(df[col])
    plt.show()

## EDA

In [ ]:
sns.barplot(data=df, x='page', y='order')

In [ ]:
sns.barplot(data=df, x='month', y='order')

In [ ]:
sns.barplot(data=df, x='day', y='order')

In [ ]:
sns.barplot(data=df, x='country', y='order')

# Preprocessing

## Feature enginerring

In [ ]:

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)

df.drop(['month', 'day'], axis=1, inplace=True)

In [ ]:
df['price_diff'] = df['price'] - df['price 2']
df['price_ratio'] = df['price'] / (df['price 2'] + 1)
df['price_sum'] = df['price'] + df['price 2']

In [ ]:
df['category_price'] = df['page 1 (main category)'] * df['price']
df['colour_category'] = df['colour'] * df['page 1 (main category)']

In [ ]:
freq = df['country'].value_counts()
df['country_freq'] = df['country'].map(freq)

In [ ]:
df['price_bin'] = pd.qcut(df['price'], 10, labels=False, duplicates='drop')

## define x and y

## y (order)
- Strong right skew (positive skew)
- to solve the issue - use log1p

In [ ]:
df['order'].value_counts()

In [ ]:
# delete year cause it the same and session id is High cardinality
X=df.drop(["order", "session ID", "year"], axis=1)
y=df['order']
y_transformed = np.log1p(y)

## Split data

In [ ]:
# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(X, y_transformed, test_size = .4,
random_state=42, shuffle =True)

## Scaling

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


# Build model

In [ ]:
input_shape = x_train. shape[1]
input_shape

In [ ]:

model = keras.models.Sequential([
    Dense(64, activation='relu', input_dim=input_shape),
    BatchNormalization(),
    Dropout(0.1),

    Dense(32, activation='relu'),
    BatchNormalization(),


    Dense(16, activation='relu'),

    Dense(1)
])


## Compile

In [ ]:

model.compile(

    optimizer=Adam(learning_rate=0.0003),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=['mae']

)



In [ ]:
model.summary()

In [ ]:

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5
)


## fit

In [ ]:

history = model.fit(
    x_train, y_train,
    validation_data=(x_test,y_test),
    epochs=50,
    batch_size=1000,
    callbacks=[early_stop,lr_scheduler]
)


# History plot

In [ ]:
def plot_history(history, figsize=(6, 12), marker='o'):
    # Get list of metrics from history (exclude val_ keys)
    metrics = [c for c in history.history.keys() if not c.startswith('val_')]

    # Create one subplot per metric
    fig, axes = plt.subplots(nrows=len(metrics), figsize=figsize, squeeze=False)
    axes = axes.flatten()  # Make axes easy to index even if len(metrics)==1

    # For each metric
    for i, metric_name in enumerate(metrics):
        ax = axes[i]

        # Get metric values from history
        train_values = history.history[metric_name]
        epochs = range(len(train_values))  # or history.epoch if available

        # Plot training metric
        ax.plot(epochs, train_values, label=metric_name, marker=marker)

        # Check if val_{metric} exists and plot validation curve
        val_metric_name = f"val_{metric_name}"
        if val_metric_name in history.history:
            val_values = history.history[val_metric_name]
            ax.plot(epochs, val_values, label=val_metric_name, marker=marker)

        # Final subplot adjustments
        ax.legend()
        ax.set_title(metric_name)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric_name)

    fig.tight_layout()
    return fig, axes

# Evaluation

## Test set

In [ ]:
y_pred_log = model.predict(x_test)
y_pred = np.expm1(y_pred_log)
model.evaluate(x_test, np.expm1(y_test))


In [ ]:
plot_history(history)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

r2_score(y_test, y_pred_log)


In [ ]:
print('Prediction items are {}'.format(y_pred[:10]))
print('Real items are {}'.format(np.expm1(y_test)[:10]))